# GNM-Contact Learner: Full Pipeline

**End-to-end:** OpenFold3 pair repr extraction → Training → Evaluation → Drive'a kaydet

**Gerekli:** Colab Pro (A100 40GB onerilir, T4 16GB de calisir)

---

## 1. Environment Setup

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 1: GPU Check & Install
# ══════════════════════════════════════════════════════════════════
!nvidia-smi
print("="*60)

# OpenFold3
!pip install -q openfold3 biopython

# Model weights (~10GB → ~/.openfold3/)
!setup_openfold

import openfold3
print(f"OpenFold3: {openfold3.__version__}")
print("="*60)

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 2: Mount Drive & Create Dirs
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
drive.mount('/content/drive')

import os, json, time, subprocess
import requests
import numpy as np
from pathlib import Path

BASE_DIR      = Path('/content/drive/MyDrive/ANM-openfold3')
PAIR_REPR_DIR = BASE_DIR / 'pair_reprs'
COORDS_DIR    = BASE_DIR / 'coords'
CKPT_DIR      = BASE_DIR / 'checkpoints'
QUERY_DIR     = Path('/content/queries')
OF3_OUT       = Path('/content/of3_output')

for d in [PAIR_REPR_DIR, COORDS_DIR, CKPT_DIR, QUERY_DIR, OF3_OUT]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Base:       {BASE_DIR}")
print(f"Pair reprs: {PAIR_REPR_DIR}")
print(f"Coords:     {COORDS_DIR}")
print(f"Checkpoints:{CKPT_DIR}")

## 2. Upload src/ Code

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 3: Write src/ modules directly into Colab
#  (Boylece GitHub'a push etmeden calisir)
# ══════════════════════════════════════════════════════════════════

SRC_DIR = Path('/content/gnm_contact_learner/src')
SRC_DIR.mkdir(parents=True, exist_ok=True)

# ─── __init__.py ───
(SRC_DIR / '__init__.py').write_text('')

# ─── ground_truth.py ───
(SRC_DIR / 'ground_truth.py').write_text('''
"""Ground truth soft contact map from PDB Ca coordinates."""
import torch

def compute_gt_probability_matrix(
    coords_ca: torch.Tensor,
    r_cut: float = 10.0,
    tau: float = 1.5,
) -> torch.Tensor:
    dist = torch.cdist(
        coords_ca.unsqueeze(0), coords_ca.unsqueeze(0)
    ).squeeze(0)
    c_gt = torch.sigmoid(-(dist - r_cut) / tau)
    c_gt.fill_diagonal_(0.0)
    return c_gt
'''.strip())

# ─── kirchhoff.py ───
(SRC_DIR / 'kirchhoff.py').write_text('''
"""Differentiable Kirchhoff matrix and GNM eigendecomposition."""
from typing import Tuple
import torch

def soft_kirchhoff(c: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    n = c.shape[-1]
    gamma = -c.clone()
    gamma.fill_diagonal_(0.0)
    diag_vals = c.sum(dim=-1)
    gamma.diagonal().copy_(diag_vals)
    gamma = gamma + eps * torch.eye(n, device=c.device, dtype=c.dtype)
    return gamma

def gnm_decompose(
    gamma: torch.Tensor, n_modes: int = 20,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    vals, vecs = torch.linalg.eigh(gamma)
    max_modes = vals.shape[-1] - 1
    k = min(n_modes, max_modes)
    eig_vals = vals[1 : k + 1]
    eig_vecs = vecs[:, 1 : k + 1]
    inv_vals = 1.0 / (eig_vals + 1e-10)
    b_factors = (eig_vecs ** 2) @ inv_vals
    return eig_vals, eig_vecs, b_factors
'''.strip())

# ─── contact_head.py ───
(SRC_DIR / 'contact_head.py').write_text('''
"""Invertible bottleneck head: pair representation <-> contact probability."""
import torch
import torch.nn as nn

class ContactProjectionHead(nn.Module):
    def __init__(self, c_z: int = 128, bottleneck_dim: int = 32) -> None:
        super().__init__()
        self.c_z = c_z
        self.bottleneck_dim = bottleneck_dim
        self.w_enc = nn.Linear(c_z, bottleneck_dim, bias=False)
        self.v = nn.Parameter(torch.randn(bottleneck_dim))
        self.w_dec = nn.Linear(bottleneck_dim, c_z, bias=False)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        z_sym = 0.5 * (z + z.transpose(1, 2))
        h = self.w_enc(z_sym)
        logits = (h * self.v).sum(dim=-1)
        logits = 0.5 * (logits + logits.transpose(-1, -2))
        n = logits.shape[-1]
        diag_mask = ~torch.eye(n, dtype=torch.bool, device=logits.device)
        logits = logits * diag_mask
        c_pred = torch.sigmoid(logits)
        c_pred = c_pred * diag_mask
        return c_pred

    def encode_bottleneck(self, z: torch.Tensor) -> torch.Tensor:
        z_sym = 0.5 * (z + z.transpose(1, 2))
        return self.w_enc(z_sym)

    @torch.no_grad()
    def inverse(self, c: torch.Tensor) -> torch.Tensor:
        c_clamped = c.clamp(1e-6, 1.0 - 1e-6)
        logit = torch.log(c_clamped / (1.0 - c_clamped))
        v_norm = self.v / (self.v.norm() + 1e-8)
        h_approx = logit.unsqueeze(-1) * v_norm
        pseudo_z = self.w_dec(h_approx)
        return pseudo_z
'''.strip())

# ─── losses.py ───
(SRC_DIR / 'losses.py').write_text('''
"""Loss functions for GNM-Contact Learner."""
from typing import Dict, Tuple
import torch
import torch.nn.functional as F
from .kirchhoff import gnm_decompose, soft_kirchhoff

def contact_loss(c_pred: torch.Tensor, c_gt: torch.Tensor, seq_sep_min: int = 6) -> torch.Tensor:
    n = c_pred.shape[-1]
    idx = torch.arange(n, device=c_pred.device)
    sep_mask = (idx.unsqueeze(0) - idx.unsqueeze(1)).abs() >= seq_sep_min
    pred_masked = c_pred[..., sep_mask]
    gt_masked = c_gt[..., sep_mask]
    pred_clamped = pred_masked.clamp(1e-7, 1.0 - 1e-7)
    return F.binary_cross_entropy(pred_clamped, gt_masked, reduction="mean")

def gnm_loss(
    c_pred: torch.Tensor, c_gt: torch.Tensor,
    n_modes: int = 20, w_eigenvalue: float = 1.0,
    w_bfactor: float = 1.0, w_eigvec: float = 0.5,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    gamma_pred = soft_kirchhoff(c_pred)
    gamma_gt = soft_kirchhoff(c_gt)
    vals_p, vecs_p, bf_p = gnm_decompose(gamma_pred, n_modes)
    vals_g, vecs_g, bf_g = gnm_decompose(gamma_gt, n_modes)
    inv_p = 1.0 / (vals_p + 1e-10)
    inv_g = 1.0 / (vals_g + 1e-10)
    inv_p_norm = inv_p / (inv_p.sum() + 1e-10)
    inv_g_norm = inv_g / (inv_g.sum() + 1e-10)
    l_eig = F.mse_loss(inv_p_norm, inv_g_norm)
    bf_p_norm = bf_p / (bf_p.max() + 1e-10)
    bf_g_norm = bf_g / (bf_g.max() + 1e-10)
    l_bf = F.mse_loss(bf_p_norm, bf_g_norm)
    cos_sim = torch.abs(F.cosine_similarity(vecs_p.T, vecs_g.T, dim=-1))
    l_vec = (1.0 - cos_sim).mean()
    loss = w_eigenvalue * l_eig + w_bfactor * l_bf + w_eigvec * l_vec
    details = {"L_eigenvalue": l_eig.item(), "L_bfactor": l_bf.item(), "L_eigvec": l_vec.item()}
    return loss, details

def reconstruction_loss(z_original: torch.Tensor, z_reconstructed: torch.Tensor) -> torch.Tensor:
    return F.mse_loss(z_reconstructed, z_original)

def total_loss(
    c_pred: torch.Tensor, c_gt: torch.Tensor,
    z_original=None, z_reconstructed=None,
    alpha: float = 1.0, beta: float = 0.5, gamma: float = 0.1,
    n_modes: int = 20,
) -> Tuple[torch.Tensor, Dict[str, float]]:
    l_contact = contact_loss(c_pred, c_gt)
    l_gnm, gnm_details = gnm_loss(c_pred, c_gt, n_modes=n_modes)
    loss = alpha * l_contact + beta * l_gnm
    details = {"L_contact": l_contact.item(), "L_gnm": l_gnm.item(), **gnm_details}
    if z_original is not None and z_reconstructed is not None:
        l_recon = reconstruction_loss(z_original, z_reconstructed)
        loss = loss + gamma * l_recon
        details["L_recon"] = l_recon.item()
    return loss, details
'''.strip())

print(f"src/ modules written to {SRC_DIR}")
!ls -la {SRC_DIR}

## 3. Training Protein List & Sequences

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 4: PDB List + Sequence Fetch
# ══════════════════════════════════════════════════════════════════

TRAINING_PDBS = [
    # Small (50-100 aa)
    "1UBQ", "1L2Y", "2JOF", "1CRN", "1VII",
    "2F4K", "1PRB", "1ENH", "2GB1", "1FME",
    # Medium (100-200 aa)
    "4AKE", "1AKE", "2LZM", "3LZM", "1HHP",
    "1MBN", "1HEL", "2RN2", "1RIS", "3CLN",
    "1CTF", "1SN3", "2CI2", "1LZ1", "1BPI",
    # Large (200-400 aa)
    "1TIM", "1GFL", "3GFP", "1HBS", "1MBO",
    "2CGA", "1CHD", "1PPT", "4HHB", "1LYZ",
    "2SOD", "1SUP", "3PGK", "1PFK", "1ADK",
    # More diverse
    "1A2P", "1BRS", "1CSE", "1DFN", "1ECA",
    "1FKJ", "1GDN", "1HMV", "1IFC", "1JWE",
]

def fetch_sequence(pdb_id: str) -> str:
    """PDB ID -> protein sequence via RCSB API."""
    url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/1"
    try:
        resp = requests.get(url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            seq = data.get("entity_poly", {}).get("pdbx_seq_one_letter_code_can", "")
            return seq.replace("\n", "")
    except Exception:
        pass
    # Fallback: FASTA
    try:
        url2 = f"https://www.rcsb.org/fasta/entry/{pdb_id}/download"
        resp2 = requests.get(url2, timeout=10)
        if resp2.status_code == 200:
            lines = resp2.text.strip().split("\n")
            return "".join(l for l in lines if not l.startswith(">"))
    except Exception:
        pass
    return ""

# Fetch all sequences
sequences = {}
for pdb_id in TRAINING_PDBS:
    seq = fetch_sequence(pdb_id)
    if seq:
        sequences[pdb_id] = seq
        print(f"  {pdb_id}: {len(seq):>4d} aa")
    else:
        print(f"  {pdb_id}: FAILED")

print(f"\nFetched: {len(sequences)}/{len(TRAINING_PDBS)}")

## 4. OpenFold3 Pair Representation Extraction

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 5: Runner YAML + Query JSONs
# ══════════════════════════════════════════════════════════════════

# Runner config: enable latent output (zij_trunk)
RUNNER_YAML = Path('/content/runner_latent.yml')
RUNNER_YAML.write_text("""
output_writer_settings:
  structure_format: cif
  write_latent_outputs: true
  write_full_confidence_scores: false
  write_features: false
""".strip())

# Create query JSONs (OpenFold3 format)
query_files = {}
for pdb_id, seq in sequences.items():
    query = {
        "queries": {
            pdb_id: {
                "chains": [{
                    "molecule_type": "protein",
                    "chain_ids": ["A"],
                    "sequence": seq,
                }]
            }
        }
    }
    qpath = QUERY_DIR / f"{pdb_id}_query.json"
    with open(qpath, 'w') as f:
        json.dump(query, f, indent=2)
    query_files[pdb_id] = qpath

print(f"Runner YAML: {RUNNER_YAML}")
print(f"Query files: {len(query_files)}")
print(f"\nwrite_latent_outputs: true  ->  zij_trunk [N, N, 128] saved")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 6: Run OpenFold3 Inference → Extract zij_trunk
#
#  --use-msa-server=True  → ColabFold MMseqs2 (~30s, no JackHMMER)
#  --runner-yaml          → write_latent_outputs: true
#  --num-diffusion-samples=1, --num-model-seeds=1  (sadece pair repr lazim)
# ══════════════════════════════════════════════════════════════════
import torch

extracted = 0
failed = []
total = len(query_files)

for i, (pdb_id, query_path) in enumerate(query_files.items()):
    # Skip if already extracted
    pair_repr_file = PAIR_REPR_DIR / f"{pdb_id}_pair_repr.pt"
    if pair_repr_file.exists():
        print(f"[{i+1}/{total}] {pdb_id}: cached")
        extracted += 1
        continue

    output_dir = OF3_OUT / pdb_id
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"[{i+1}/{total}] {pdb_id}: running...", end="", flush=True)
    t0 = time.time()

    try:
        result = subprocess.run([
            'run_openfold', 'predict',
            f'--query-json={query_path}',
            f'--output-dir={output_dir}',
            f'--runner-yaml={RUNNER_YAML}',
            '--use-msa-server=True',
            '--use-templates=True',
            '--num-diffusion-samples=1',
            '--num-model-seeds=1',
        ], capture_output=True, text=True, timeout=900)

        elapsed = time.time() - t0

        if result.returncode != 0:
            print(f" FAIL ({elapsed:.0f}s)")
            err = result.stderr[-300:] if result.stderr else "no stderr"
            print(f"    {err}")
            failed.append(pdb_id)
            continue

        # Find latent output
        latent_files = list(output_dir.rglob("*_latent_output.pt"))
        if not latent_files:
            print(f" no latent file ({elapsed:.0f}s)")
            failed.append(pdb_id)
            continue

        latent = torch.load(latent_files[0], map_location='cpu')

        # Extract zij_trunk (pair representation)
        zij = latent.get('zij_trunk')
        si = latent.get('si_trunk')

        if zij is None:
            # Try alternative key names
            for key in latent:
                t = latent[key]
                if isinstance(t, torch.Tensor) and t.dim() == 4:
                    zij = t
                    print(f" (used key '{key}')", end="")
                    break

        if zij is not None:
            torch.save({
                'pair_repr': zij,
                'single_repr': si,
                'pdb_id': pdb_id,
                'n_tokens': zij.shape[-3],
                'c_z': zij.shape[-1],
                'sequence': sequences[pdb_id],
            }, pair_repr_file)
            extracted += 1
            print(f" OK  zij={list(zij.shape)} ({elapsed:.0f}s)")
        else:
            print(f" zij_trunk not found. keys={list(latent.keys())}")
            failed.append(pdb_id)

    except subprocess.TimeoutExpired:
        print(f" TIMEOUT")
        failed.append(pdb_id)
    except Exception as e:
        print(f" ERROR: {e}")
        failed.append(pdb_id)

print(f"\n{'='*60}")
print(f"Extracted: {extracted}/{total}")
if failed:
    print(f"Failed ({len(failed)}): {failed}")

## 5. Ground Truth Coordinates

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 7: Download PDB -> Ca Coordinates
# ══════════════════════════════════════════════════════════════════
import Bio.PDB as bpdb

parser = bpdb.PDBParser(QUIET=True)
pdbl = bpdb.PDBList()
PDB_CACHE = Path('/content/pdb_cache')
PDB_CACHE.mkdir(exist_ok=True)

saved = 0
for pdb_id in sequences:
    coord_file = COORDS_DIR / f"{pdb_id}_ca.pt"
    if coord_file.exists():
        saved += 1
        continue
    try:
        pdb_file = pdbl.retrieve_pdb_file(
            pdb_id, pdir=str(PDB_CACHE), file_format='pdb'
        )
        structure = parser.get_structure(pdb_id, pdb_file)
        first_chain = list(structure[0].get_chains())[0]
        ca_coords = []
        for res in first_chain:
            if res.get_id()[0] == ' ' and 'CA' in res:
                ca_coords.append(res['CA'].get_vector().get_array())
        if ca_coords:
            coords = torch.tensor(np.array(ca_coords), dtype=torch.float32)
            torch.save(coords, coord_file)
            saved += 1
            print(f"  {pdb_id}: {len(ca_coords)} Ca")
    except Exception as e:
        print(f"  {pdb_id}: {e}")

print(f"\nCoords: {saved}/{len(sequences)}")

## 6. Data Verification

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 8: Verify & Build Dataset Index
# ══════════════════════════════════════════════════════════════════
import torch

pair_files = sorted(PAIR_REPR_DIR.glob("*_pair_repr.pt"))
coord_files = sorted(COORDS_DIR.glob("*_ca.pt"))

pair_ids = {f.stem.replace('_pair_repr', '') for f in pair_files}
coord_ids = {f.stem.replace('_ca', '') for f in coord_files}
matched = sorted(pair_ids & coord_ids)

print(f"Pair repr files: {len(pair_files)}")
print(f"Coord files:     {len(coord_files)}")
print(f"Matched:         {len(matched)}")
print()

# Detect c_z dimension from first file
C_Z = 128  # default
if pair_files:
    sample = torch.load(pair_files[0], map_location='cpu')
    zij = sample['pair_repr']
    C_Z = zij.shape[-1]
    print(f"Detected c_z = {C_Z}")
    print(f"Sample: {sample['pdb_id']} zij={list(zij.shape)}")
    if sample.get('single_repr') is not None:
        print(f"         si={list(sample['single_repr'].shape)}")

# Quick check: sizes match?
print(f"\nSize check (first 5):")
for pdb_id in matched[:5]:
    pr = torch.load(PAIR_REPR_DIR / f"{pdb_id}_pair_repr.pt", map_location='cpu')
    ca = torch.load(COORDS_DIR / f"{pdb_id}_ca.pt", map_location='cpu')
    n_tok = pr['pair_repr'].shape[-3]
    n_ca = ca.shape[0]
    status = "OK" if n_tok >= n_ca else "WARN: n_tok < n_ca"
    print(f"  {pdb_id}: n_token={n_tok}, n_ca={n_ca}  {status}")

## 7. Training

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 9: Dataset + DataLoader
# ══════════════════════════════════════════════════════════════════
import sys
sys.path.insert(0, '/content/gnm_contact_learner')

import torch
from torch.utils.data import Dataset, DataLoader, random_split
from src.ground_truth import compute_gt_probability_matrix


class CachedPairReprDataset(Dataset):
    """Dataset from pre-extracted pair_repr .pt + coord .pt files."""

    def __init__(self, pdb_ids, pair_dir, coord_dir, r_cut=10.0, tau=1.5):
        self.pdb_ids = pdb_ids
        self.pair_dir = Path(pair_dir)
        self.coord_dir = Path(coord_dir)
        self.r_cut = r_cut
        self.tau = tau

    def __len__(self):
        return len(self.pdb_ids)

    def __getitem__(self, idx):
        pdb_id = self.pdb_ids[idx]

        # Load pair repr
        pr_data = torch.load(
            self.pair_dir / f"{pdb_id}_pair_repr.pt",
            map_location='cpu', weights_only=False
        )
        zij = pr_data['pair_repr']  # [1, N_tok, N_tok, c_z] or [N_tok, N_tok, c_z]

        # Ensure [1, N, N, c_z]
        if zij.dim() == 3:
            zij = zij.unsqueeze(0)

        # Load coords
        coords = torch.load(
            self.coord_dir / f"{pdb_id}_ca.pt",
            map_location='cpu', weights_only=True
        )  # [N_ca, 3]

        # Align sizes: n_token may differ from n_ca
        n_tok = zij.shape[1]
        n_ca = coords.shape[0]
        N = min(n_tok, n_ca)
        zij = zij[:, :N, :N, :]
        coords = coords[:N]

        # Ground truth contact map
        c_gt = compute_gt_probability_matrix(coords, r_cut=self.r_cut, tau=self.tau)

        return {
            'pair_repr': zij,      # [1, N, N, c_z]
            'c_gt': c_gt,          # [N, N]
            'coords_ca': coords,   # [N, 3]
            'pdb_id': pdb_id,
        }


# Build dataset
dataset = CachedPairReprDataset(matched, PAIR_REPR_DIR, COORDS_DIR)
print(f"Dataset size: {len(dataset)}")

# Train/val split (85/15)
n_val = max(1, int(len(dataset) * 0.15))
n_train = len(dataset) - n_val
train_ds, val_ds = random_split(
    dataset, [n_train, n_val],
    generator=torch.Generator().manual_seed(42)
)

# DataLoader (batch_size=1 because proteins have different sizes)
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False)

print(f"Train: {n_train}, Val: {n_val}")

# Quick test
sample = dataset[0]
print(f"\nSample: {sample['pdb_id']}")
print(f"  pair_repr: {list(sample['pair_repr'].shape)}")
print(f"  c_gt:      {list(sample['c_gt'].shape)}")
print(f"  coords:    {list(sample['coords_ca'].shape)}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 10: Full Training Loop
# ══════════════════════════════════════════════════════════════════
from src.contact_head import ContactProjectionHead
from src.losses import total_loss
from src.kirchhoff import soft_kirchhoff, gnm_decompose

# ── Hyperparameters ──
EPOCHS = 100
LR = 1e-4
WEIGHT_DECAY = 1e-2
ALPHA = 1.0      # L_contact weight
BETA = 0.5       # L_gnm weight
GAMMA = 0.1      # L_recon weight
N_MODES = 20
GRAD_CLIP = 1.0
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f"Device: {DEVICE}")
print(f"c_z: {C_Z}")
print(f"Epochs: {EPOCHS}, LR: {LR}")
print(f"Loss weights: alpha={ALPHA}, beta={BETA}, gamma={GAMMA}")
print()

# ── Model ──
head = ContactProjectionHead(c_z=C_Z, bottleneck_dim=32).to(DEVICE)
n_params = sum(p.numel() for p in head.parameters())
print(f"ContactProjectionHead: {n_params:,} parameters")

optimizer = torch.optim.AdamW(head.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)


def pearson_corr(x, y):
    xc = x - x.mean()
    yc = y - y.mean()
    return ((xc * yc).sum() / (xc.norm() * yc.norm()).clamp(min=1e-8)).item()


# ── Training ──
best_val_loss = float('inf')
history = []

for epoch in range(EPOCHS):
    # ── Train ──
    head.train()
    train_loss_sum = 0.0
    train_n = 0

    for batch in train_loader:
        z = batch['pair_repr'].to(DEVICE)    # [1, 1, N, N, c_z] from DataLoader
        c_gt = batch['c_gt'].to(DEVICE)      # [1, N, N]

        # Remove DataLoader batch dim if nested
        if z.dim() == 5:
            z = z.squeeze(0)
        if c_gt.dim() == 3:
            c_gt = c_gt.squeeze(0)

        c_pred = head(z).squeeze(0)  # [N, N]

        # Reconstruction
        z_sym = 0.5 * (z + z.transpose(1, 2))
        h = head.w_enc(z_sym)
        z_recon = head.w_dec(h)

        loss, details = total_loss(
            c_pred, c_gt,
            z_original=z_sym, z_reconstructed=z_recon,
            alpha=ALPHA, beta=BETA, gamma=GAMMA, n_modes=N_MODES,
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(head.parameters(), GRAD_CLIP)
        optimizer.step()

        train_loss_sum += loss.item()
        train_n += 1

    scheduler.step()
    avg_train = train_loss_sum / max(train_n, 1)

    # ── Validate ──
    head.eval()
    val_loss_sum = 0.0
    val_acc_sum = 0.0
    val_bf_sum = 0.0
    val_n = 0

    with torch.no_grad():
        for batch in val_loader:
            z = batch['pair_repr'].to(DEVICE)
            c_gt = batch['c_gt'].to(DEVICE)
            if z.dim() == 5:
                z = z.squeeze(0)
            if c_gt.dim() == 3:
                c_gt = c_gt.squeeze(0)

            c_pred = head(z).squeeze(0)
            loss, _ = total_loss(c_pred, c_gt, alpha=ALPHA, beta=BETA, n_modes=N_MODES)

            # Adjacency accuracy
            acc = ((c_pred > 0.5) == (c_gt > 0.5)).float().mean().item()

            # B-factor Pearson
            gamma_p = soft_kirchhoff(c_pred)
            gamma_g = soft_kirchhoff(c_gt)
            _, _, bf_p = gnm_decompose(gamma_p, N_MODES)
            _, _, bf_g = gnm_decompose(gamma_g, N_MODES)
            bf_corr = pearson_corr(bf_p, bf_g)

            val_loss_sum += loss.item()
            val_acc_sum += acc
            val_bf_sum += bf_corr
            val_n += 1

    avg_val = val_loss_sum / max(val_n, 1)
    avg_acc = val_acc_sum / max(val_n, 1)
    avg_bf = val_bf_sum / max(val_n, 1)

    history.append({
        'epoch': epoch + 1,
        'train_loss': avg_train,
        'val_loss': avg_val,
        'val_adj_acc': avg_acc,
        'val_bf_pearson': avg_bf,
    })

    # Save best
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': head.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': avg_val,
            'val_adj_acc': avg_acc,
            'val_bf_pearson': avg_bf,
            'c_z': C_Z,
            'bottleneck_dim': 32,
            'config': {
                'alpha': ALPHA, 'beta': BETA, 'gamma': GAMMA,
                'lr': LR, 'n_modes': N_MODES,
            },
        }, CKPT_DIR / 'best_model.pt')

    # Log
    if (epoch + 1) % 5 == 0 or epoch == 0:
        star = ' *' if avg_val <= best_val_loss else ''
        print(
            f"Epoch {epoch+1:3d}/{EPOCHS}  "
            f"train={avg_train:.4f}  "
            f"val={avg_val:.4f}  "
            f"acc={avg_acc:.3f}  "
            f"bf_r={avg_bf:.3f}{star}"
        )

# Save final
torch.save({
    'epoch': EPOCHS,
    'model_state_dict': head.state_dict(),
    'c_z': C_Z,
    'bottleneck_dim': 32,
    'history': history,
}, CKPT_DIR / 'final_model.pt')

print(f"\nTraining complete!")
print(f"Best val loss: {best_val_loss:.4f}")
print(f"Checkpoints saved to: {CKPT_DIR}")

## 8. Evaluation & Visualization

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 11: Training Curves
# ══════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

epochs = [h['epoch'] for h in history]
train_losses = [h['train_loss'] for h in history]
val_losses = [h['val_loss'] for h in history]
val_accs = [h['val_adj_acc'] for h in history]
val_bfs = [h['val_bf_pearson'] for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, train_losses, label='Train')
axes[0].plot(epochs, val_losses, label='Val')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Total Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, val_accs, color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Val Adjacency Accuracy')
axes[1].set_ylim(0, 1)
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, val_bfs, color='orange')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Pearson r')
axes[2].set_title('Val B-factor Correlation')
axes[2].set_ylim(-0.2, 1)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'training_curves.png'), dpi=150)
plt.show()
print(f"Saved: {CKPT_DIR / 'training_curves.png'}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 12: Contact Map Visualization (Best Model)
# ══════════════════════════════════════════════════════════════════

# Load best model
ckpt = torch.load(CKPT_DIR / 'best_model.pt', map_location=DEVICE)
head_eval = ContactProjectionHead(c_z=ckpt['c_z'], bottleneck_dim=ckpt['bottleneck_dim']).to(DEVICE)
head_eval.load_state_dict(ckpt['model_state_dict'])
head_eval.eval()

print(f"Loaded best model (epoch {ckpt['epoch']}, val_loss={ckpt['val_loss']:.4f})")

# Visualize first 3 val samples
n_show = min(3, len(val_ds))
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
if n_show == 1:
    axes = axes[None, :]  # ensure 2D

with torch.no_grad():
    for i in range(n_show):
        sample = val_ds[i]
        z = sample['pair_repr'].to(DEVICE)
        if z.dim() == 3:
            z = z.unsqueeze(0)
        c_gt = sample['c_gt'].numpy()
        c_pred = head_eval(z).squeeze().cpu().numpy()
        pdb_id = sample['pdb_id']

        axes[i, 0].imshow(c_gt, cmap='hot', vmin=0, vmax=1)
        axes[i, 0].set_title(f'{pdb_id} - Ground Truth')

        axes[i, 1].imshow(c_pred, cmap='hot', vmin=0, vmax=1)
        axes[i, 1].set_title(f'{pdb_id} - Predicted')

        diff = np.abs(c_pred - c_gt)
        axes[i, 2].imshow(diff, cmap='Blues', vmin=0, vmax=0.5)
        axes[i, 2].set_title(f'{pdb_id} - |Diff|')

plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'contact_maps.png'), dpi=150)
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 13: B-factor Profile Comparison
# ══════════════════════════════════════════════════════════════════

n_show = min(3, len(val_ds))
fig, axes = plt.subplots(1, n_show, figsize=(5 * n_show, 4))
if n_show == 1:
    axes = [axes]

with torch.no_grad():
    for i in range(n_show):
        sample = val_ds[i]
        z = sample['pair_repr'].to(DEVICE)
        if z.dim() == 3:
            z = z.unsqueeze(0)
        c_gt = sample['c_gt'].to(DEVICE)
        c_pred = head_eval(z).squeeze(0)
        pdb_id = sample['pdb_id']

        # GNM B-factors
        _, _, bf_gt = gnm_decompose(soft_kirchhoff(c_gt), N_MODES)
        _, _, bf_pred = gnm_decompose(soft_kirchhoff(c_pred), N_MODES)

        bf_gt_n = (bf_gt / bf_gt.max()).cpu().numpy()
        bf_pred_n = (bf_pred / bf_pred.max()).cpu().numpy()
        r = pearson_corr(bf_pred.cpu(), bf_gt.cpu())

        axes[i].plot(bf_gt_n, label='GT', linewidth=2)
        axes[i].plot(bf_pred_n, label='Pred', linewidth=2, linestyle='--')
        axes[i].set_title(f'{pdb_id}  (r={r:.3f})')
        axes[i].set_xlabel('Residue')
        axes[i].set_ylabel('Normalized B-factor')
        axes[i].legend()
        axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(CKPT_DIR / 'bfactor_profiles.png'), dpi=150)
plt.show()

## 9. Final Summary & Export

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 14: Full Evaluation on All Val Samples
# ══════════════════════════════════════════════════════════════════

results = []
head_eval.eval()

with torch.no_grad():
    for i in range(len(val_ds)):
        sample = val_ds[i]
        z = sample['pair_repr'].to(DEVICE)
        if z.dim() == 3:
            z = z.unsqueeze(0)
        c_gt = sample['c_gt'].to(DEVICE)
        c_pred = head_eval(z).squeeze(0)

        # Metrics
        acc = ((c_pred > 0.5) == (c_gt > 0.5)).float().mean().item()
        _, _, bf_p = gnm_decompose(soft_kirchhoff(c_pred), N_MODES)
        _, _, bf_g = gnm_decompose(soft_kirchhoff(c_gt), N_MODES)
        bf_r = pearson_corr(bf_p.cpu(), bf_g.cpu())

        loss, details = total_loss(c_pred, c_gt, alpha=ALPHA, beta=BETA, n_modes=N_MODES)

        results.append({
            'pdb_id': sample['pdb_id'],
            'n_residues': c_gt.shape[0],
            'adj_acc': acc,
            'bf_pearson': bf_r,
            'loss': loss.item(),
            **details,
        })

# Summary
print("="*70)
print("VALIDATION RESULTS")
print("="*70)
print(f"{'PDB':>6s} {'N':>4s} {'Acc':>6s} {'BF_r':>6s} {'Loss':>8s} {'L_cont':>8s} {'L_gnm':>8s}")
print("-"*70)
for r in results:
    print(
        f"{r['pdb_id']:>6s} {r['n_residues']:4d} "
        f"{r['adj_acc']:6.3f} {r['bf_pearson']:6.3f} "
        f"{r['loss']:8.4f} {r['L_contact']:8.4f} {r['L_gnm']:8.4f}"
    )

# Averages
avg_acc = np.mean([r['adj_acc'] for r in results])
avg_bf = np.mean([r['bf_pearson'] for r in results])
avg_loss = np.mean([r['loss'] for r in results])
print("-"*70)
print(f"{'AVG':>6s} {'':>4s} {avg_acc:6.3f} {avg_bf:6.3f} {avg_loss:8.4f}")
print("="*70)

# Save results
import json
with open(CKPT_DIR / 'val_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved: {CKPT_DIR / 'val_results.json'}")

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  CELL 15: Drive Summary
# ══════════════════════════════════════════════════════════════════

print("Files saved to Google Drive:")
print(f"  {CKPT_DIR / 'best_model.pt'}")
print(f"  {CKPT_DIR / 'final_model.pt'}")
print(f"  {CKPT_DIR / 'training_curves.png'}")
print(f"  {CKPT_DIR / 'contact_maps.png'}")
print(f"  {CKPT_DIR / 'bfactor_profiles.png'}")
print(f"  {CKPT_DIR / 'val_results.json'}")
print()
print(f"  {PAIR_REPR_DIR}/  ({len(list(PAIR_REPR_DIR.glob('*.pt')))} files)")
print(f"  {COORDS_DIR}/    ({len(list(COORDS_DIR.glob('*.pt')))} files)")
print()
print("To use locally:")
print("  1. Download best_model.pt")
print("  2. Load with:")
print("     from src.contact_head import ContactProjectionHead")
print("     head = ContactProjectionHead(c_z=128, bottleneck_dim=32)")
print("     ckpt = torch.load('best_model.pt', map_location='cpu')")
print("     head.load_state_dict(ckpt['model_state_dict'])")
print("  3. Inverse path (no OpenFold3 needed):")
print("     from src.inverse import PairReprFromCoords")
print("     gen = PairReprFromCoords(head)")
print("     pseudo_z = gen.from_pdb('protein.pdb')")